# CMC 87708 — Primary Detector-state comparison (Gemma + Qwen, 768)

Reproduces the revised-manuscript primary comparison:
- 3 datasets × 500 held-out targets
- Basic vs Source-gated on identical complete Detector-state packets
- Gemma-2-9B-IT and Qwen2.5-7B-Instruct
- deterministic decoding, max_new_tokens=768
- protected detector-state audit, paired statistics, and AlignScore

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ramita99/cmc_llm_87708.git
%cd cmc_llm_87708
!pip install -q -e .

In [ ]:
import json, yaml, shutil, subprocess, sys
from pathlib import Path
import pandas as pd
import torch
from google.colab import userdata
from huggingface_hub import login, snapshot_download, hf_hub_download

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("BF16-capable GPU required")

hf_token = userdata.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

SOURCE_ROOT = Path("/content/drive/MyDrive/CMC_87708_primary_packets")
DATASETS = ["NF-UNSW-NB15-v3","NF-CSE-CIC-IDS2018-v3","NF-ToN-IoT-v3"]

MODELS = {
    "gemma": {
        "id":"google/gemma-2-9b-it",
        "revision":"11c9b309abf73637e4b6f9a3fa1e92e615547819",
        "root":Path("/content/drive/MyDrive/CMC_87708_primary_gemma"),
    },
    "qwen": {
        "id":"Qwen/Qwen2.5-7B-Instruct",
        "revision":"a09a35458c702b33eeacc393d103063234e8bc28",
        "root":Path("/content/drive/MyDrive/CMC_87708_primary_qwen"),
    },
}

def load_jsonl(p):
    with Path(p).open("r",encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

for spec in MODELS.values():
    for sub in ["packets","manifests","results"]:
        (spec["root"]/sub).mkdir(parents=True,exist_ok=True)
    manifest=[]
    for ds in DATASETS:
        source=[p for p in load_jsonl(SOURCE_ROOT/"packets"/f"{ds}.jsonl") if p.get("state")=="detector"]
        assert len(source)==500,(ds,len(source))
        with (spec["root"]/"packets"/f"{ds}.jsonl").open("w",encoding="utf-8") as f:
            for p in source:
                f.write(json.dumps(p,ensure_ascii=False,default=str)+"\n")
                t=p["traceability"]
                manifest.append({"dataset":ds,"window_id":t.get("window_id"),"edge_id":t.get("edge_id")})
    pd.DataFrame(manifest).to_csv(spec["root"]/"manifests"/"target_manifest.csv",index=False)

In [ ]:
for label,spec in MODELS.items():
    local_model=snapshot_download(repo_id=spec["id"],revision=spec["revision"],token=hf_token)
    cfg={
        "protocol_version":f"CMC-87708-PRIMARY-{label}-768",
        "output_root":str(spec["root"]),
        "sample_seed":7,
        "targets_per_dataset":500,
        "states":["detector"],
        "methods":["basic_prompt","proposed_source_gated"],
        "full_state_for_method_comparison":"detector",
        "generation_dtype":"bfloat16",
        "attention_implementation":"sdpa",
        "generation_batch_size":4,
        "max_new_tokens":768,
        "do_sample":False,
        "numeric_tolerance":0.0001,
        "fallback_on_truncation":True,
        "bootstrap_replicates":10000,
        "multiple_testing_correction":"holm",
    }
    cfg_path=Path(f"/content/{label}_primary_768.yaml")
    cfg_path.write_text(yaml.safe_dump(cfg,sort_keys=False),encoding="utf-8")

    subprocess.run([
        sys.executable,"scripts/02_run_primary_generation.py",
        "--experiment",str(cfg_path),"--model",str(local_model),
        "--methods","basic_prompt","proposed_source_gated","--resume"
    ],check=True)
    subprocess.run([sys.executable,"scripts/03_recompute_primary_metrics.py","--experiment",str(cfg_path)],check=True)
    subprocess.run([sys.executable,"scripts/04_primary_paired_statistics.py","--experiment",str(cfg_path)],check=True)

print("Primary generation and exact-field statistics complete.")

In [ ]:
# Deterministic reference rendering (1,500 detector packets; model-independent safeguard/reference)
spec=MODELS["gemma"]
cfg_path=Path("/content/gemma_primary_768.yaml")
subprocess.run([
    sys.executable,"scripts/02_run_primary_generation.py",
    "--experiment",str(cfg_path),
    "--methods","deterministic_template","--resume"
],check=True)

In [ ]:
# AlignScore-large, nli_sp
ckpt=hf_hub_download(repo_id="yzha/AlignScore",filename="AlignScore-large.ckpt",token=hf_token)
for label,spec in MODELS.items():
    subprocess.run([
        sys.executable,"scripts/05_primary_alignscore.py",
        "--root",str(spec["root"]),
        "--label",f"{label}_primary_768",
        "--checkpoint",ckpt,
        "--batch-size","32",
        "--device","cuda",
        "--bootstrap-replicates","10000",
        "--seed","7",
    ],check=True)
print("Primary AlignScore complete.")